# Tutorial 10 — Reliable evaluation, calibration, and responsible reporting

[Lecture notes](../lectures/10_evaluation.md) · [Exercise sheet](../exercises/10_evaluation.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Calculate Brier score, reliability bins, coverage/accuracy tradeoffs, and a simulated judge-order audit.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Accuracy and proper scoring

The probabilities are invented binary predictions. Calibration exercises need probability semantics; a retrieval cosine score cannot simply be substituted.

Before running: predict the output or the invariant being checked.


In [2]:
y=np.array([1,0,1,0,1,0,1,0,1,0])
p=np.array([.95,.8,.7,.1,.4,.2,.6,.55,.9,.35])
pred=(p>=.5).astype(int); conf=np.maximum(p,1-p); correct=(pred==y)
print("accuracy",correct.mean(),"binary Brier",np.mean((p-y)**2))
assert np.isclose(np.mean((np.array([.8,.7,.4,.1])-np.array([1,0,1,0]))**2),.225)


accuracy 0.7 binary Brier 0.17375000000000002


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Reliability and selective prediction

Bins refer to confidence in the predicted class. Empty bins are omitted from ECE and recorded with count zero.

Before running: predict the output or the invariant being checked.


In [3]:
edges=np.linspace(0,1,6); ece=0
for j,(lo,hi) in enumerate(zip(edges[:-1],edges[1:])):
    mask=(conf>=lo)&((conf<hi) if j<4 else (conf<=hi))
    if mask.any():
        acc=correct[mask].mean(); c=conf[mask].mean()
        ece+=mask.mean()*abs(acc-c)
        print("bin",lo,hi,"n",mask.sum(),"accuracy",acc,"confidence",c)
    else: print("bin",lo,hi,"n",0)
print("ECE",ece)
for threshold in [.5,.7,.9,.99]:
    keep=conf>=threshold
    print("threshold",threshold,"coverage",keep.mean(),"selective accuracy",float(correct[keep].mean()) if keep.any() else None)


bin 0.0 0.2 n 0
bin 0.2 0.4 n 0
bin 0.4 0.6000000000000001 n 3 accuracy 0.3333333333333333 confidence 0.5833333333333334
bin 0.6000000000000001 0.8 n 2 accuracy 1.0 confidence 0.675
bin 0.8 1.0 n 5 accuracy 0.8 confidence 0.8700000000000001
ECE 0.17500000000000004
threshold 0.5 coverage 1.0 selective accuracy 0.7
threshold 0.7 coverage 0.6 selective accuracy 0.8333333333333334
threshold 0.9 coverage 0.3 selective accuracy 1.0
threshold 0.99 coverage 0.0 selective accuracy None


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Audit order-sensitive judging

Synthetic judge choices are expressed as system identities after accounting for display order. A flip under swapping is a warning sign, not a full bias diagnosis.

Before running: predict the output or the invariant being checked.


In [4]:
normal=np.array(["A","A","B","A","B","A"])
swapped=np.array(["B","A","A","B","B","A"])
print("identity preference changes after swap",np.mean(normal!=swapped))
assert np.mean(normal!=swapped)==.5
# Paired uncertainty for accuracy uses evaluation items, not probability bins.
idx=rng.integers(0,len(y),size=(1000,len(y)))
print("accuracy bootstrap interval",np.quantile(correct[idx].mean(axis=1),[.025,.975]))


identity preference changes after swap 0.5
accuracy bootstrap interval [0.4 0.9]


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Can two equally accurate models differ in calibration?
2. Why must an abstention threshold be chosen on development data?
3. What would reveal position bias in a model judge?


## Extension and submission

Apply the evaluation functions to real outputs from earlier model extensions. Use the included JSONL result schema to preserve IDs and provenance.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
